# Sales CSV - Phase validation

This notebook collects read-only evidence for the complete CSV Medallion pipeline. It checks layer counts, rejected records, enrichment results, Gold outputs, business invariants, and metadata without modifying the ETL tables.

In [0]:
dbutils.widgets.removeAll()

## Validation configuration

The environment widget resolves the same Bronze, Silver, rejected, and Gold objects used by the pipeline, ensuring that the evidence belongs to the selected deployment.

In [0]:
dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

environment = dbutils.widgets.get("environment").lower()

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "catalog": "salescsv_dev"
    },
    "prod": {
        "catalog": "salescsv_prod"
    }
}

catalog = config[environment]["catalog"]

product_bronze_table = f"{catalog}.bronze.product_catalog_raw"
inventory_bronze_table = f"{catalog}.bronze.inventory_transactions_raw"

silver_table = f"{catalog}.silver.inventory_movements"
rejected_table = f"{catalog}.silver.rejected_transactions"

product_gold_table = f"{catalog}.gold.inventory_by_product"
warehouse_gold_table = f"{catalog}.gold.inventory_by_warehouse"
low_stock_gold_table = f"{catalog}.gold.low_stock_products"

print("=" * 60)
print("SALES CSV - PHASE VALIDATION")
print("=" * 60)
print(f"Environment : {environment}")
print(f"Catalog     : {catalog}")
print("=" * 60)

## Completeness and data-quality evidence

Counts show how the two Bronze sources flow into accepted and rejected Silver outcomes. Rejection samples retain raw values and reasons, while accepted samples demonstrate product enrichment produced by the Silver LEFT JOIN.

In [0]:
product_bronze_count = spark.table(
    product_bronze_table
).count()

inventory_bronze_count = spark.table(
    inventory_bronze_table
).count()

silver_count = spark.table(
    silver_table
).count()

rejected_count = spark.table(
    rejected_table
).count()

product_gold_count = spark.table(
    product_gold_table
).count()

warehouse_gold_count = spark.table(
    warehouse_gold_table
).count()

low_stock_count = spark.table(
    low_stock_gold_table
).count()

print("=" * 60)
print("CSV MEDALLION SUMMARY")
print("=" * 60)
print(f"Bronze products          : {product_bronze_count}")
print(f"Bronze transactions      : {inventory_bronze_count}")
print(f"Silver valid             : {silver_count}")
print(f"Silver rejected          : {rejected_count}")
print(f"Gold product rows        : {product_gold_count}")
print(f"Gold warehouse rows      : {warehouse_gold_count}")
print(f"Gold low-stock rows      : {low_stock_count}")
print("=" * 60)

In [0]:
display(
    spark.table(rejected_table)
        .select(
            "transaction_id",
            "product_id",
            "warehouse",
            "transaction_type",
            "quantity_raw",
            "rejection_reason"
        )
        .orderBy("transaction_id")
)
display(
    spark.table(rejected_table)
        .groupBy("rejection_reason")
        .count()
        .orderBy("rejection_reason")
)

In [0]:
display(
    spark.table(silver_table)
        .select(
            "transaction_id",
            "product_id",
            "product_name",
            "category",
            "supplier",
            "warehouse",
            "transaction_type",
            "quantity",
            "inventory_change",
            "inventory_value_change"
        )
        .orderBy("transaction_id")
        .limit(30)
)

## Gold outputs and cross-layer reconciliation

Ordered product, warehouse, and low-stock views provide reviewable analytical evidence. The following checks reconcile total inventory units between Silver and Gold and assert that every low-stock row satisfies the published exception rule.

In [0]:
display(
    spark.table(product_gold_table)
        .orderBy("current_stock")
)
display(
    spark.table(warehouse_gold_table)
        .orderBy("inventory_value", ascending=False)
)
display(
    spark.table(low_stock_gold_table)
        .orderBy("current_stock")
)

In [0]:
from pyspark.sql.functions import sum, round

silver_units = (
    spark.table(silver_table)
        .agg(
            sum("inventory_change").alias("units")
        )
        .first()["units"]
)

gold_units = (
    spark.table(product_gold_table)
        .agg(
            sum("current_stock").alias("units")
        )
        .first()["units"]
)

silver_value = (
    spark.table(silver_table)
        .agg(
            round(
                sum("inventory_value_change"),
                2
            ).alias("value")
        )
        .first()["value"]
)

gold_value = (
    spark.table(product_gold_table)
        .agg(
            round(
                sum("inventory_value"),
                2
            ).alias("value")
        )
        .first()["value"]
)

print("=" * 60)
print("CSV GOLD RECONCILIATION")
print("=" * 60)
print(f"Silver net units        : {silver_units}")
print(f"Gold net units          : {gold_units}")
print()
print(f"Silver inventory value  : {silver_value}")
print(f"Gold inventory value    : {gold_value}")
print()

print(
    "Units reconciliation   : "
    + ("PASS" if silver_units == gold_units else "FAIL")
)

print(
    "Value reconciliation   : "
    + ("PASS" if silver_value == gold_value else "FAIL")
)

print("=" * 60)

In [0]:
invalid_low_stock_count = (
    spark.table(low_stock_gold_table)
        .filter(
            "current_stock > reorder_level"
        )
        .count()
)

if invalid_low_stock_count == 0:
    print(
        "PASS - All low-stock records satisfy "
        "current_stock <= reorder_level."
    )
else:
    print(
        f"FAIL - Found {invalid_low_stock_count} "
        "invalid low-stock records."
    )

## Catalog documentation evidence

The final inspection confirms that expected table comments are present in Unity Catalog, linking technical validation with the metadata contract applied by notebook `98_metadata_documentation`.

In [0]:
tables_to_validate = [
    product_bronze_table,
    inventory_bronze_table,
    silver_table,
    rejected_table,
    product_gold_table,
    warehouse_gold_table,
    low_stock_gold_table
]

for table_name in tables_to_validate:

    print(f"\n===== {table_name} =====")

    display(
        spark.sql(
            f"DESCRIBE EXTENDED {table_name}"
        ).filter(
            "col_name IN ('Type', 'Provider', 'Location', 'Comment')"
        )
    )